# 01 — ACEA PDF files Download

This notebook downloads the ACEA monthly passenger car registration PDF reports or excel files used as the primary source for the project.

The reports contain monthly new car registrations by European market and power source, including:

- Battery electric
- Plug-in hybrid
- Hybrid electric
- Petrol
- Diesel
- Other power sources
- Total registrations

The downloaded PDFs are stored in the raw data folder and used in the next notebook for extraction and parsing.

In [ ]:
%pip install pdfplumber

# Full Download script 

In [ ]:
import re
import itertools
import calendar
from pathlib import Path

import pandas as pd
import pdfplumber
import requests


In [ ]:
# ----------------------------
# PATH CONFIGURATION
# ----------------------------

# Assumes notebook is run from /notebooks directory
PROJECT_ROOT = Path.cwd().parent

PROJECT_ROOT = Path.cwd().parent
BRONZE_DIR = PROJECT_ROOT / "data" / "bronze" / "acea_pdfs"

In [ ]:
# ----------------------------
# DOWNLOAD PARAMETERS
# ----------------------------
YEAR = 2024

In [ ]:
# ------------------------------------------------------------
# 1) DOWNLOAD WITH FULL LOGGING
# ------------------------------------------------------------
def download_pdf(url: str, out_dir: Path = BRONZE_DIR) -> Path:
    
    def unique(seq):
        seen = set()
        return [x for x in seq if not (x in seen or seen.add(x))]

    # Build URL variants
    candidate_urls = unique([
        url,

        # Pattern 1
        url.replace("_January_", "-January_")
           .replace("_February_", "-February_")
           .replace("_March_", "-March_")
           .replace("_April_", "-April_")
           .replace("_May_", "-May_")
           .replace("_June_", "-June_")
           .replace("_July_", "-July_")
           .replace("_August_", "-August_")
           .replace("_September_", "-September_")
           .replace("_October_", "-October_")
           .replace("_November_", "-November_")
           .replace("_December_", "-December_"),

        # Pattern 2
        url.replace("_January_", "_January-")
           .replace("_February_", "_February-")
           .replace("_March_", "_March-")
           .replace("_April_", "_April-")
           .replace("_May_", "_May-")
           .replace("_June_", "_June-")
           .replace("_July_", "_July-")
           .replace("_August_", "_August-")
           .replace("_September_", "_September-")
           .replace("_October_", "_October-")
           .replace("_November_", "_November-")
           .replace("_December_", "_December-"),
    ])

    out_dir.mkdir(parents=True, exist_ok=True)

    last_error = None

    for test_url in candidate_urls:
        try:
            print(f"Trying: {test_url}")

            r = requests.get(test_url, timeout=60)
            r.raise_for_status()

            filename = test_url.split("/")[-1]
            pdf_path = out_dir / filename
            pdf_path.write_bytes(r.content)

            print(f"✅ Downloaded: {filename}")
            return pdf_path

        except requests.exceptions.HTTPError as e:
            print(f"❌ Failed: {test_url}")
            last_error = e

    raise ValueError(f"All URL attempts failed for base URL: {url}") from last_error

import calendar


results = []

for month_num in range(1, 13):
    month_label = calendar.month_name[month_num]
    url = f"https://www.acea.auto/files/Press_release_car_registrations_{month_label}_{YEAR}.pdf"

    try:
        pdf_path = download_pdf(url)
        results.append((month_label, "OK"))

    except Exception:
        print(f"❌ {month_label} FAILED")
        results.append((month_label, "FAILED"))

# Summary
print("\n=== SUMMARY ===")

failed = [m for m, status in results if status == "FAILED"]

if not failed:
    print("All 12 files downloaded successfully ✅")
else:
    print(f"Failed months: {failed}")


## Output

This notebook produces a local collection of ACEA monthly PDF reports.

These files are used as raw source inputs in the next step:

**02_acea_registration_parsing.ipynb**